## Member 7 – Correctness Verification

**Goal:** check whether each parallel version gives the same result as the **sequential baseline**.

| Version | Notebook | Expected result |
|---|---|---|
| Unsafe (no synchronization) | Parallel code (Member 3) | Incorrect / inconsistent |
| Lock / critical section | Safe Parallel code (Member 4) | Same as sequential |
| Local accumulation / reduction | Local Accumulation (Member 5) | Same as sequential |

Both shared resources from the group's experiments are verified:
1. **Shared output pixel** – 3 workers add their kernel-row partial sums to one output pixel (1000 repeats).
2. **Shared counter** – 2 workers increment one counter (50,000 iterations each).

The worker functions below are copied unchanged from the group notebooks. Each version is run **10 times**, because a race condition depends on thread timing and may not show the same error every run.

In [1]:
import threading
import time
import numpy as np
from PIL import Image

image_path = "test_images/large_1024x1024.png"
image = np.array(Image.open(image_path).convert("L"), dtype=np.float64)

gaussian_kernel = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=np.float64) / 16

N_RUNS = 10
N_PIXEL_WORKERS, REPEATS = 3, 1000
N_COUNTER_WORKERS, ITERATIONS = 2, 50000

In [2]:
# Partial sums for the centre pixel (same as the group notebooks)
def get_partial_sums(n_workers=N_PIXEL_WORKERS):
    i, j = image.shape[0] // 2, image.shape[1] // 2
    partial_sums = []
    for k in range(n_workers):
        partial = 0.0
        for l in range(3):
            partial += image[i + k - 1, j + l - 1] * gaussian_kernel[k, l]
        partial_sums.append(partial)
    return partial_sums

partial_sums = get_partial_sums()
print("Partial sums:", [float(p) for p in partial_sums])

Partial sums: [3.8125, 9.125, 5.125]


### 1. Sequential baseline (single thread, no sharing)

In [3]:
def sequential_pixel():
    out = 0.0
    for partial in partial_sums:
        for _ in range(REPEATS):
            out += partial
    return out

def sequential_counter():
    counter = 0
    for _ in range(N_COUNTER_WORKERS * ITERATIONS):
        counter += 1
    return counter

t0 = time.perf_counter(); SEQ_PIXEL = sequential_pixel(); t_seq_pixel = time.perf_counter() - t0
t0 = time.perf_counter(); SEQ_COUNTER = sequential_counter(); t_seq_counter = time.perf_counter() - t0
print("Sequential pixel result  :", SEQ_PIXEL, f"({t_seq_pixel:.6f} s)")
print("Sequential counter result:", SEQ_COUNTER, f"({t_seq_counter:.6f} s)")

Sequential pixel result  : 18062.5 (0.000452 s)
Sequential counter result: 100000 (0.008075 s)


### 2. Worker functions (copied from Members 3, 4 and 5)

In [4]:
# --- Member 3: unsafe ---
def pixel_worker(shared_output, partial_sum, repeats):
    for _ in range(repeats):
        temp = shared_output[0]
        time.sleep(0)            # for thread switching
        shared_output[0] = temp + partial_sum

def counter_worker(counter, iterations):
    for _ in range(iterations):
        temp = counter[0]
        time.sleep(0)
        counter[0] = temp + 1

# --- Member 4: lock / critical section ---
def safe_pixel_worker(shared_output, partial_sum, repeats, lock):
    for _ in range(repeats):
        with lock:  # Critical Section
            temp = shared_output[0]
            shared_output[0] = temp + partial_sum

def safe_counter_worker(counter, iterations, lock):
    for _ in range(iterations):
        with lock:  # Critical Section
            counter[0] += 1

# --- Member 5: local accumulation / reduction ---
def local_pixel_worker(local_results, worker_idx, partial_sum, repeats):
    local_acc = 0.0
    for _ in range(repeats):
        local_acc += partial_sum
    local_results[worker_idx] = local_acc

def local_counter_worker(local_counts, worker_idx, iterations):
    local_count = 0
    for _ in range(iterations):
        local_count += 1
    local_counts[worker_idx] = local_count

### 3. Run each version and return its final value

In [5]:
def run_threads(threads):
    start = time.perf_counter()
    for t in threads: t.start()
    for t in threads: t.join()
    return time.perf_counter() - start

def pixel_unsafe():
    out = [0.0]
    t = run_threads([threading.Thread(target=pixel_worker, args=(out, p, REPEATS)) for p in partial_sums])
    return out[0], t

def pixel_lock():
    out, lock = [0.0], threading.Lock()
    t = run_threads([threading.Thread(target=safe_pixel_worker, args=(out, p, REPEATS, lock)) for p in partial_sums])
    return out[0], t

def pixel_reduction():
    local = [0.0] * len(partial_sums)
    t0 = time.perf_counter()
    run_threads([threading.Thread(target=local_pixel_worker, args=(local, k, p, REPEATS)) for k, p in enumerate(partial_sums)])
    value = sum(local)                      # reduction step
    return value, time.perf_counter() - t0

def counter_unsafe():
    c = [0]
    t = run_threads([threading.Thread(target=counter_worker, args=(c, ITERATIONS)) for _ in range(N_COUNTER_WORKERS)])
    return c[0], t

def counter_lock():
    c, lock = [0], threading.Lock()
    t = run_threads([threading.Thread(target=safe_counter_worker, args=(c, ITERATIONS, lock)) for _ in range(N_COUNTER_WORKERS)])
    return c[0], t

def counter_reduction():
    local = [0] * N_COUNTER_WORKERS
    t0 = time.perf_counter()
    run_threads([threading.Thread(target=local_counter_worker, args=(local, k, ITERATIONS)) for k in range(N_COUNTER_WORKERS)])
    value = sum(local)                      # reduction step
    return value, time.perf_counter() - t0

### 4. Verify against the sequential baseline

In [6]:
def verify(name, fn, expected, runs=N_RUNS):
    values, times = [], []
    for _ in range(runs):
        v, t = fn()
        values.append(v); times.append(t)
    correct = sum(abs(v - expected) < 1e-9 for v in values)
    errors = [abs(expected - v) for v in values]
    return {
        "version": name,
        "expected": expected,
        "min": min(values), "max": max(values),
        "mean_error": sum(errors) / runs,
        "correct_runs": f"{correct}/{runs}",
        "mean_time": sum(times) / runs,
        "verdict": "CORRECT" if correct == runs else "INCORRECT",
    }

results = [
    verify("Pixel - Unsafe",    pixel_unsafe,      SEQ_PIXEL),
    verify("Pixel - Lock",      pixel_lock,        SEQ_PIXEL),
    verify("Pixel - Reduction", pixel_reduction,   SEQ_PIXEL),
    verify("Counter - Unsafe",    counter_unsafe,    SEQ_COUNTER),
    verify("Counter - Lock",      counter_lock,      SEQ_COUNTER),
    verify("Counter - Reduction", counter_reduction, SEQ_COUNTER),
]

hdr = f"{'Version':<22}{'Expected':>11}{'Min':>11}{'Max':>11}{'Mean error':>12}{'Correct':>9}{'Mean time (s)':>15}  Verdict"
print(hdr); print("-" * len(hdr))
for r in results:
    print(f"{r['version']:<22}{r['expected']:>11g}{r['min']:>11g}{r['max']:>11g}"
          f"{r['mean_error']:>12.2f}{r['correct_runs']:>9}{r['mean_time']:>15.6f}  {r['verdict']}")

Version                  Expected        Min        Max  Mean error  Correct  Mean time (s)  Verdict
----------------------------------------------------------------------------------------------------
Pixel - Unsafe            18062.5     3812.5    9975.19    12009.69     0/10       0.004582  INCORRECT
Pixel - Lock              18062.5    18062.5    18062.5        0.00    10/10       0.001685  CORRECT
Pixel - Reduction         18062.5    18062.5    18062.5        0.00    10/10       0.000702  CORRECT
Counter - Unsafe           100000      50000      50754    49890.00     0/10       0.056886  INCORRECT
Counter - Lock             100000     100000     100000        0.00    10/10       0.025959  CORRECT
Counter - Reduction        100000     100000     100000        0.00    10/10       0.005243  CORRECT


### 5. Conclusion

- The **unsafe** version does not match the sequential baseline: updates to the shared pixel and the shared counter are lost, and the final value changes from run to run. This confirms a race condition (read–modify–write on shared data without synchronization).
- The **lock** version and the **local accumulation / reduction** version match the sequential baseline in every run, so both synchronization methods are correct.
- Note on timing: the unsafe workers call `time.sleep(0)` inside the loop to force thread switching, so their execution time is not a fair speed comparison with the safe versions.